# Video → frames → cleaning/dedup → preprocessing (Spark + OpenCV)

Этот ноутбук делает три первые стадии пайплайна:

1. Нарезка видео на кадры  
2. Очистка и дедупликация  
3. Предобработка (crop, resize, normalization)

Подход рассчитан на локальный учебный стек: Spark/HDFS/Hive/NiFi локально в Docker, а notebook — как точка оркестрации.

## Почему так

Для **одного локального видео** невыгодно декодировать `.mp4` прямо внутри Spark-задач.  
Практичнее:

- **OpenCV / FFmpeg** — извлечь кадры из видео
- **Spark** — обработать уже извлечённые кадры параллельно:
  - посчитать метрики качества
  - убрать мусор / дубликаты
  - подготовить изображения для модели

Это хорошо ложится на вашу архитектуру пайплайна.

In [ ]:
# Если запускаете в отдельном окружении, раскомментируйте при необходимости:
# !pip install pyspark opencv-python pillow numpy pandas

In [ ]:
import os
import io
import cv2
import math
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from pyspark.sql import SparkSession, functions as F, types as T, Window

In [ ]:
# --- НАСТРОЙКИ ---

# 1) Укажите путь к вашему локальному видео
VIDEO_PATH = "/path/to/your/video.mp4"

# 2) Базовая папка проекта
BASE_DIR = Path("/mnt/data/video_pipeline_demo")

RAW_FRAMES_DIR = BASE_DIR / "staging" / "frames_raw"
PROCESSED_FRAMES_DIR = BASE_DIR / "processed" / "frames"
META_DIR = BASE_DIR / "meta"

# 3) Параметры извлечения кадров
EXTRACT_EVERY_N_FRAMES = 10      # брать каждый N-й кадр
MAX_FRAMES = None                # можно ограничить для теста, например 500

# 4) Параметры очистки
MIN_LAPLACIAN_VAR = 50.0         # ниже -> размыто
MIN_BRIGHTNESS = 25.0            # слишком тёмные кадры удаляем
MAX_BRIGHTNESS = 245.0           # слишком пересвеченные кадры удаляем

# 5) Параметры дедупликации
# Для последовательного видео со статичной камерой удобно сравнивать соседние кадры по dHash
HAMMING_THRESHOLD = 4            # чем меньше, тем строже дедупликация

# 6) Параметры предобработки
# ROI в долях изображения: (x1, y1, x2, y2)
# Здесь можно отрезать верх/низ или лишние области сцены
CROP = (0.05, 0.10, 0.95, 0.90)

TARGET_W = 640
TARGET_H = 640

# 7) Очистить старые результаты?
RESET_OUTPUT = False

In [ ]:
# Подготовка папок
if RESET_OUTPUT and BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)

for p in [RAW_FRAMES_DIR, PROCESSED_FRAMES_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR =", BASE_DIR)
print("RAW_FRAMES_DIR =", RAW_FRAMES_DIR)
print("PROCESSED_FRAMES_DIR =", PROCESSED_FRAMES_DIR)

In [ ]:
# Инициализация Spark
spark = (
    SparkSession.builder
    .appName("video-frames-cleaning-preprocessing")
    .master("spark://spark-master:7077")   # если notebook видит ваш spark-master в docker-сети
    # .master("local[*]")                  # если хотите сначала просто локально протестировать
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark

## Шаг 1. Нарезка видео на кадры

Для учебного проекта это лучше делать на драйвере через OpenCV.  
Результат: папка `staging/frames_raw`, где каждый файл — отдельный кадр.

In [ ]:
def extract_frames(
    video_path: str,
    output_dir: Path,
    every_n_frames: int = 10,
    max_frames: int | None = None
):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Видео не найдено: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    print(f"FPS={fps:.3f}, total_frames={total_frames}, size={width}x{height}")

    saved = []
    frame_idx = 0
    saved_count = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % every_n_frames == 0:
            ts_sec = frame_idx / fps if fps else None
            filename = f"frame_{frame_idx:08d}.jpg"
            filepath = output_dir / filename

            cv2.imwrite(str(filepath), frame, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

            saved.append({
                "frame_idx": int(frame_idx),
                "timestamp_sec": float(ts_sec) if ts_sec is not None else None,
                "path": str(filepath),
                "width": int(frame.shape[1]),
                "height": int(frame.shape[0]),
            })
            saved_count += 1

            if max_frames is not None and saved_count >= max_frames:
                break

        frame_idx += 1

    cap.release()

    df = pd.DataFrame(saved)
    return df, {
        "fps": fps,
        "total_frames": total_frames,
        "width": width,
        "height": height,
        "saved_frames": saved_count,
    }

In [ ]:
frames_pd, video_info = extract_frames(
    video_path=VIDEO_PATH,
    output_dir=RAW_FRAMES_DIR,
    every_n_frames=EXTRACT_EVERY_N_FRAMES,
    max_frames=MAX_FRAMES
)

print(video_info)
frames_pd.head()

In [ ]:
frames_pd.to_parquet(META_DIR / "extracted_frames.parquet", index=False)
len(frames_pd)

## Шаг 2. Очистка и дедупликация

Что делаем:

- считаем для каждого кадра:
  - яркость
  - резкость (`variance of Laplacian`)
  - `dHash`
- выкидываем:
  - слишком тёмные / пересвеченные кадры
  - слишком размытые кадры
  - почти одинаковые соседние кадры

In [ ]:
frames_spark = spark.createDataFrame(frames_pd)
frames_spark.printSchema()
frames_spark.show(5, truncate=False)

In [ ]:
@F.udf(returnType=T.DoubleType())
def brightness_udf(path: str):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return float(np.mean(img))
    except Exception:
        return None


@F.udf(returnType=T.DoubleType())
def laplacian_var_udf(path: str):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return float(cv2.Laplacian(img, cv2.CV_64F).var())
    except Exception:
        return None


def _compute_dhash(path: str) -> str | None:
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None

        resized = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
        diff = resized[:, 1:] > resized[:, :-1]

        bits = ''.join('1' if x else '0' for x in diff.flatten())
        hex_str = hex(int(bits, 2))[2:].rjust(16, '0')
        return hex_str
    except Exception:
        return None


@F.udf(returnType=T.StringType())
def dhash_udf(path: str):
    return _compute_dhash(path)


@F.udf(returnType=T.IntegerType())
def hamming_udf(h1: str, h2: str):
    try:
        if h1 is None or h2 is None:
            return None
        b1 = bin(int(h1, 16))[2:].zfill(64)
        b2 = bin(int(h2, 16))[2:].zfill(64)
        return sum(c1 != c2 for c1, c2 in zip(b1, b2))
    except Exception:
        return None

In [ ]:
metrics_df = (
    frames_spark
    .withColumn("brightness", brightness_udf("path"))
    .withColumn("laplacian_var", laplacian_var_udf("path"))
    .withColumn("dhash", dhash_udf("path"))
)

metrics_df.show(5, truncate=False)

In [ ]:
# Базовая фильтрация по качеству
quality_df = (
    metrics_df
    .withColumn(
        "quality_ok",
        (
            (F.col("brightness").between(MIN_BRIGHTNESS, MAX_BRIGHTNESS)) &
            (F.col("laplacian_var") >= F.lit(MIN_LAPLACIAN_VAR)) &
            F.col("dhash").isNotNull()
        )
    )
)

quality_df.groupBy("quality_ok").count().show()

In [ ]:
# Дедупликация относительно предыдущего валидного кадра
# Для статичной камеры и плавного движения объекта это простой и рабочий baseline.
w = Window.orderBy("frame_idx")

dedup_df = (
    quality_df
    .filter(F.col("quality_ok") == True)
    .withColumn("prev_dhash", F.lag("dhash").over(w))
    .withColumn("hamming_to_prev", hamming_udf("dhash", "prev_dhash"))
    .withColumn(
        "is_duplicate",
        F.when(F.col("prev_dhash").isNull(), F.lit(False))
         .when(F.col("hamming_to_prev") <= F.lit(HAMMING_THRESHOLD), F.lit(True))
         .otherwise(F.lit(False))
    )
)

dedup_df.select(
    "frame_idx", "timestamp_sec", "brightness", "laplacian_var", "dhash", "hamming_to_prev", "is_duplicate"
).show(20, truncate=False)

In [ ]:
clean_df = dedup_df.filter(F.col("is_duplicate") == False).cache()

print("Всего извлечено кадров:", frames_spark.count())
print("После quality filter:", quality_df.filter(F.col("quality_ok") == True).count())
print("После дедупликации:", clean_df.count())

In [ ]:
clean_pd = clean_df.orderBy("frame_idx").toPandas()
clean_pd.to_parquet(META_DIR / "clean_frames.parquet", index=False)
clean_pd.head()

## Шаг 3. Предобработка

Сейчас делаем:

- crop по ROI
- resize до `TARGET_W x TARGET_H`
- нормализация в диапазон `[0, 1]`
- сохранение JPEG в `processed/frames`

> Для инференса модели часто сохраняют уже `uint8 JPEG/PNG`, а нормализацию выполняют в dataloader.  
> Но для демонстрации здесь считаем и нормализованный тензор.

In [ ]:
def preprocess_and_save(
    input_path: str,
    output_dir: Path,
    crop: tuple[float, float, float, float],
    target_w: int,
    target_h: int
):
    img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Не удалось прочитать: {input_path}")

    h, w = img.shape[:2]
    x1r, y1r, x2r, y2r = crop

    x1 = max(0, min(w - 1, int(w * x1r)))
    y1 = max(0, min(h - 1, int(h * y1r)))
    x2 = max(x1 + 1, min(w, int(w * x2r)))
    y2 = max(y1 + 1, min(h, int(h * y2r)))

    cropped = img[y1:y2, x1:x2]
    resized = cv2.resize(cropped, (target_w, target_h), interpolation=cv2.INTER_AREA)

    normalized = resized.astype(np.float32) / 255.0

    out_name = Path(input_path).name
    out_path = output_dir / out_name

    # Сохраняем обработанный кадр как JPEG
    cv2.imwrite(str(out_path), resized, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

    return {
        "processed_path": str(out_path),
        "crop_x1": x1, "crop_y1": y1, "crop_x2": x2, "crop_y2": y2,
        "target_w": target_w, "target_h": target_h,
        "norm_min": float(normalized.min()),
        "norm_max": float(normalized.max()),
        "norm_mean": float(normalized.mean()),
    }

In [ ]:
def process_partition(rows_iter):
    for row in rows_iter:
        row = row.asDict()
        try:
            meta = preprocess_and_save(
                input_path=row["path"],
                output_dir=PROCESSED_FRAMES_DIR,
                crop=CROP,
                target_w=TARGET_W,
                target_h=TARGET_H
            )
            yield {
                **row,
                **meta,
                "status": "ok",
                "error": None
            }
        except Exception as e:
            yield {
                **row,
                "processed_path": None,
                "crop_x1": None, "crop_y1": None, "crop_x2": None, "crop_y2": None,
                "target_w": None, "target_h": None,
                "norm_min": None, "norm_max": None, "norm_mean": None,
                "status": "error",
                "error": str(e)
            }

In [ ]:
processed_schema = T.StructType([
    T.StructField("frame_idx", T.LongType(), True),
    T.StructField("timestamp_sec", T.DoubleType(), True),
    T.StructField("path", T.StringType(), True),
    T.StructField("width", T.LongType(), True),
    T.StructField("height", T.LongType(), True),
    T.StructField("brightness", T.DoubleType(), True),
    T.StructField("laplacian_var", T.DoubleType(), True),
    T.StructField("dhash", T.StringType(), True),
    T.StructField("quality_ok", T.BooleanType(), True),
    T.StructField("prev_dhash", T.StringType(), True),
    T.StructField("hamming_to_prev", T.IntegerType(), True),
    T.StructField("is_duplicate", T.BooleanType(), True),
    T.StructField("processed_path", T.StringType(), True),
    T.StructField("crop_x1", T.IntegerType(), True),
    T.StructField("crop_y1", T.IntegerType(), True),
    T.StructField("crop_x2", T.IntegerType(), True),
    T.StructField("crop_y2", T.IntegerType(), True),
    T.StructField("target_w", T.IntegerType(), True),
    T.StructField("target_h", T.IntegerType(), True),
    T.StructField("norm_min", T.DoubleType(), True),
    T.StructField("norm_max", T.DoubleType(), True),
    T.StructField("norm_mean", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("error", T.StringType(), True),
])

In [ ]:
processed_rdd = clean_df.orderBy("frame_idx").rdd.mapPartitions(process_partition)
processed_df = spark.createDataFrame(processed_rdd, schema=processed_schema).cache()

processed_df.select(
    "frame_idx", "processed_path", "status", "norm_mean"
).show(20, truncate=False)

In [ ]:
processed_df.write.mode("overwrite").parquet(str(META_DIR / "processed_frames_parquet"))
processed_df.toPandas().to_parquet(META_DIR / "processed_frames.parquet", index=False)

print("Готово.")
print("Processed frames dir:", PROCESSED_FRAMES_DIR)
print("Metadata dir:", META_DIR)

## Визуальная проверка нескольких кадров

In [ ]:
import matplotlib.pyplot as plt

sample_paths = [p for p in sorted(PROCESSED_FRAMES_DIR.glob("*.jpg"))[:6]]

for p in sample_paths:
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(p.name)
    plt.axis("off")
    plt.show()

## Что улучшить дальше

### 1. Дедупликация не только по соседнему кадру
Можно сравнивать:
- не только с предыдущим,
- но и с предыдущими `K` кадрами,
- или кластеризовать по hash/embedding.

### 2. Более умный crop
Так как камера статична, можно:
- один раз задать ROI вручную,
- либо найти область движения через background subtraction,
- либо выделять область корпуса корабля детектором.

### 3. Запись в HDFS
После локальной отладки можно писать:
- `raw video` → `hdfs:///raw/...`
- `frames_raw` → `hdfs:///staging/...`
- `processed frames` → `hdfs:///processed/...`

### 4. Связка с Hive
Отдельно можно завести таблицы:
- `video_frames_metadata`
- `video_frames_processed`
- `defect_detections`

## Мини-итог

Для вашего кейса с **одной статичной камерой** и проходом корабля мимо дока стартовый вариант такой:

- извлекаем каждый `N`-й кадр,
- фильтруем мусор по blur/brightness,
- убираем похожие соседние кадры,
- режем ROI,
- приводим к размеру модели,
- сохраняем подготовленные кадры и метаданные.

Это хороший baseline для следующего шага — детекции дефектов.